## Example submission

Image Matching Challenge 2025: https://www.kaggle.com/competitions/image-matching-challenge-2025

This notebook creates a simple submission using ALIKED and LightGlue, plus DINO for shortlisting, on GPU. Adapted from [last year](https://www.kaggle.com/code/oldufo/imc-2024-submission-example).

Remember to select an accelerator on the sidebar to the right, and to disable internet access when submitting a notebook to the competition.

In [1]:
# IMPORTANT 
#Install dependencies and copy model weights to run the notebook without internet access when submitting to the competition.

!pip install --no-index /kaggle/input/imc2024-packages-lightglue-rerun-kornia/* --no-deps
!mkdir -p /root/.cache/torch/hub/checkpoints
!cp /kaggle/input/aliked/pytorch/aliked-n16/1/aliked-n16.pth /root/.cache/torch/hub/checkpoints/
!cp /kaggle/input/lightglue/pytorch/aliked/1/aliked_lightglue.pth /root/.cache/torch/hub/checkpoints/
!cp /kaggle/input/lightglue/pytorch/aliked/1/aliked_lightglue.pth /root/.cache/torch/hub/checkpoints/aliked_lightglue_v0-1_arxiv-pth

Processing /kaggle/input/imc2024-packages-lightglue-rerun-kornia/kornia-0.7.2-py2.py3-none-any.whl
Processing /kaggle/input/imc2024-packages-lightglue-rerun-kornia/kornia_moons-0.2.9-py3-none-any.whl
Processing /kaggle/input/imc2024-packages-lightglue-rerun-kornia/kornia_rs-0.1.2-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
Processing /kaggle/input/imc2024-packages-lightglue-rerun-kornia/lightglue-0.0-py3-none-any.whl
Processing /kaggle/input/imc2024-packages-lightglue-rerun-kornia/pycolmap-0.6.1-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
Processing /kaggle/input/imc2024-packages-lightglue-rerun-kornia/rerun_sdk-0.15.0a2-cp38-abi3-manylinux_2_31_x86_64.whl
  Attempting uninstall: kornia-rs
    Found existing installation: kornia_rs 0.1.8
    Uninstalling kornia_rs-0.1.8:
      Successfully uninstalled kornia_rs-0.1.8
  Attempting uninstall: kornia
    Found existing installation: kornia 0.8.0
    Uninstalling kornia-0.8.0:
      Successfully uninstalled ko

In [2]:
!python -m pip install --no-index --find-links=/kaggle/input/pkg-check-orientation check_orientation==0.0.5 > /dev/null
!mkdir -p /root/.cache/torch/hub/checkpoints/
!cp /kaggle/input/pkg-check-orientation/2020-11-16_resnext50_32x4d.zip /root/.cache/torch/hub/checkpoints/

In [3]:
import sys
import os
from tqdm import tqdm
from time import time, sleep
import gc
import numpy as np
import h5py
import dataclasses
import pandas as pd
from IPython.display import clear_output
from collections import defaultdict
from copy import deepcopy
from PIL import Image

import cv2
import torch
import torch.nn.functional as F
import kornia as K
import kornia.feature as KF

import torch
from lightglue import match_pair
from lightglue import ALIKED, LightGlue
from lightglue.utils import load_image, rbd
from transformers import AutoImageProcessor, AutoModel

# IMPORTANT Utilities: importing data into colmap and competition metric
import pycolmap
sys.path.append('/kaggle/input/imc25-utils')
from database import *
from h5_to_db import *
import metric

print("============ Fine tuning Parametesr ==============")

# Do not forget to select an accelerator on the sidebar to the right.
device = K.utils.get_cuda_device_if_available(0)
print(f'{device=}')

/usr/local/lib/python3.10/dist-packages/kornia/feature/lightglue.py:44: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_fwd(cast_inputs=torch.float32)
/usr/local/lib/python3.10/dist-packages/lightglue/lightglue.py:24: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_fwd(cast_inputs=torch.float32)


============ Fine tuning Parametesr ==============
device=device(type='cuda', index=0)


In [4]:
from check_orientation.pre_trained_models import create_model
from torchvision.io import read_image as T_read_image
from torchvision.io import ImageReadMode
from torchvision import transforms
from typing import Callable, Tuple, List, Optional, Any
from torch.utils.data import Dataset, DataLoader


def convert_rot_k(index: int) -> int:
    """
    根据输入的索引返回对应的旋转值。
    0 -> 0, 1 -> 3, 2 -> 2, 其他 -> 1
    """
    mapping = [0, 3, 2]
    if 0 <= index < len(mapping):
        return mapping[index]
    return 1


class CheckRotationDataset(Dataset):
    """
    数据集：用于检查图像旋转。
    """
    def __init__(
        self, 
        file_paths: List[str], 
        transform: Optional[Callable[[Any], Any]] = None
    ) -> None:
        """
        初始化数据集。

        Args:
            file_paths (List[str]): 图像文件路径列表。
            transform (Optional[Callable]): 图像变换函数，默认为 None。
        """
        self.file_paths = file_paths
        self.transform = transform

    def __len__(self) -> int:
        """返回数据集大小。"""
        return len(self.file_paths)

    def __getitem__(self, index: int) -> Any:
        """根据索引获取图像并进行转换。"""
        file_path = self.file_paths[index]
        image = T_read_image(file_path, mode=ImageReadMode.RGB)
        return self.transform(image) if self.transform else image


def create_rotation_dataloader(
    image_paths: List[str], 
    batch_size: int = 1
) -> DataLoader:
    """
    构建用于旋转检查的数据加载器。

    Args:
        image_paths (List[str]): 图像路径列表
        batch_size (int): 批量大小

    Returns:
        DataLoader: 数据加载器实例
    """
    # 定义图像预处理变换
    preprocessing = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ConvertImageDtype(torch.float),
        transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
    ])

    # 创建数据集
    rotation_dataset = CheckRotationDataset(
        file_paths=image_paths, 
        transform=preprocessing
    )

    # 构建数据加载器
    return DataLoader(
        dataset=rotation_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=True,
        drop_last=False
    )


def execute_rotation_detection(
    image_files: List[str], 
    device: str
) -> List[int]:
    """
    对图像文件列表执行旋转检测。

    Args:
        image_files (List[str]): 图像文件路径列表
        device (str): 计算设备，如 'cuda' 或 'cpu'

    Returns:
        List[int]: 每张图片对应的旋转类别
    """
    # 加载模型并设置为评估模式
    model = create_model("swsl_resnext50_32x4d")
    model.to(device)
    model.eval()

    # 构建数据加载器
    dataloader = create_rotation_dataloader(image_files)

    rotations = []

    for idx, images in enumerate(dataloader):
        images = images.to(torch.float32).to(device)
        try:
            with torch.no_grad():
                logits = model(images)
                predictions = logits.detach().cpu().numpy()
                detected_rotation = predictions[0].argmax()
                rotation_k = convert_rot_k(detected_rotation)
                rotations.append(rotation_k)
                image_name = os.path.basename(image_files[idx])
                print(f"{image_name} > rot_k={rotation_k}")
        except Exception as e:
            print(f"Error processing {image_files[idx]}: {e}")
            rotations.append(None)
    
    return rotations

def convert_coord(
    coords: np.ndarray, 
    width: int, 
    height: int, 
    rotation_k: int
) -> np.ndarray:
    """
    根据旋转参数 rotk，对坐标进行变换。

    Args:
        coords (np.ndarray): 形状为 (N, 2) 的二维坐标数组。
        width (int): 图像宽度。
        height (int): 图像高度。
        rotation_k (int): 旋转类别（0、1、2、3）。

    Returns:
        np.ndarray: 变换后的坐标，形状与输入一致。
    """

    def rot0(r):
        return r

    def rot1(r):
        x = width - 1 - r[:, 1]
        y = r[:, 0]
        return np.column_stack((x, y))

    def rot2(r):
        x = width - 1 - r[:, 0]
        y = height - 1 - r[:, 1]
        return np.column_stack((x, y))

    def rot3(r):
        x = r[:, 1]
        y = height - 1 - r[:, 0]
        return np.column_stack((x, y))

    rotation_funcs: dict[int, Callable[[np.ndarray], np.ndarray]] = {
        0: rot0,
        1: rot1,
        2: rot2,
        3: rot3
    }

    return rotation_funcs.get(rotation_k, rot0)(coords)

/usr/local/lib/python3.10/dist-packages/timm/models/_factory.py:117: UserWarning: Mapping deprecated model name swsl_resnext50_32x4d to current resnext50_32x4d.fb_swsl_ig1b_ft_in1k.
  model = create_fn(


In [5]:
def load_torch_image(fname, device=torch.device('cpu')):
    img = K.io.load_image(fname, K.io.ImageLoadType.RGB32, device=device)[None, ...]
    return img


# Must Use efficientnet global descriptor to get matching shortlists.
def get_global_desc(fnames, device = torch.device('cpu')):
    processor = AutoImageProcessor.from_pretrained('/kaggle/input/dinov2/pytorch/base/1')
    model = AutoModel.from_pretrained('/kaggle/input/dinov2/pytorch/base/1')
    model = model.eval()
    model = model.to(device)
    global_descs_dinov2 = []
    for i, img_fname_full in tqdm(enumerate(fnames),total= len(fnames)):
        key = os.path.splitext(os.path.basename(img_fname_full))[0]
        timg = load_torch_image(img_fname_full)
        with torch.inference_mode():
            inputs = processor(images=timg, return_tensors="pt", do_rescale=False).to(device)
            outputs = model(**inputs)
            dino_mac = F.normalize(outputs.last_hidden_state[:,1:].max(dim=1)[0], dim=1, p=2)
        global_descs_dinov2.append(dino_mac.detach().cpu())
    global_descs_dinov2 = torch.cat(global_descs_dinov2, dim=0)
    return global_descs_dinov2


def get_img_pairs_exhaustive(img_fnames):
    index_pairs = []
    for i in range(len(img_fnames)):
        for j in range(i+1, len(img_fnames)):
            index_pairs.append((i,j))
    return index_pairs


def get_image_pairs_shortlist(fnames,
                              sim_th = 0.6, # should be strict
                              min_pairs = 30,
                              exhaustive_if_less = 20,
                              device=torch.device('cpu')):
    num_imgs = len(fnames)
    if num_imgs <= exhaustive_if_less:
        return get_img_pairs_exhaustive(fnames)
    descs = get_global_desc(fnames, device=device)
    dm = torch.cdist(descs, descs, p=2).detach().cpu().numpy()
    # removing half
    mask = dm <= sim_th
    total = 0
    matching_list = []
    ar = np.arange(num_imgs)
    already_there_set = []
    for st_idx in range(num_imgs-1):
        mask_idx = mask[st_idx]
        to_match = ar[mask_idx]
        if len(to_match) < min_pairs:
            to_match = np.argsort(dm[st_idx])[:min_pairs]  
        for idx in to_match:
            if st_idx == idx:
                continue
            if dm[st_idx, idx] < 1000:
                matching_list.append(tuple(sorted((st_idx, idx.item()))))
                total+=1
    matching_list = sorted(list(set(matching_list)))
    return matching_list

from pathlib import Path
from typing import List
import torch
import h5py
from tqdm import tqdm

def detect_aliked(
    image_paths: List[str],
    rotations,
    output_dir: str = ".featureout",
    max_features: int = 4096,
    resize_dim: int = 2048,
    device: torch.device = torch.device('cpu')
) -> None:
    """
    利用 ALIKED 检测关键点和描述子，并保存至指定文件夹。

    Args:
        image_paths (List[str]): 图像路径列表。
        rotations (List[int]): 每张图像的旋转角度（rot_k）。
        output_dir (str): 特征保存目录。
        max_features (int): 最大关键点数量。
        resize_dim (int): 图像调整大小。
        device (torch.device): 运算设备。
    """

    dtype = torch.float32  # ALIKED 不支持 float16
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)

    # 初始化特征提取器
    extractor = ALIKED(
        max_num_keypoints=max_features,
        detection_threshold=0.2,
        resize=resize_dim
    ).eval().to(device, dtype)

    # 打开HDF5文件用于写入
    with h5py.File(str(output_path / "keypoints.h5"), "w") as f_kp, \
         h5py.File(str(output_path / "keypoints_rot.h5"), "w") as f_kp_rot, \
         h5py.File(str(output_path / "descriptors.h5"), "w") as f_desc:

        for img_path, rot_k in tqdm(zip(image_paths, rotations), total=len(image_paths)):
            image_name = Path(img_path).name
            key = image_name

            try:
                with torch.inference_mode():
                    img_tensor = load_torch_image(img_path, device=device).to(dtype)
                    height, width = img_tensor.shape[2], img_tensor.shape[3]
                    rotated_img = torch.rot90(img_tensor, rot_k, [2, 3])

                    features = extractor.extract(rotated_img)
                    keypoints = features['keypoints'].reshape(-1, 2).cpu().numpy()
                    descriptors = features['descriptors'].reshape(len(keypoints), -1).cpu().numpy()

                    # 保存旋转后坐标
                    f_kp_rot[key] = keypoints
                    # 坐标反变换到原图
                    original_keypoints = convert_coord(keypoints, width, height, rot_k)
                    f_kp[key] = original_keypoints
                    # 保存描述子
                    f_desc[key] = descriptors

            except Exception as e:
                print(f"Failed processing {key}: {e}")

def match_with_lightglue(
    image_paths: List[str],
    pair_indices: List[Tuple[int, int]],
    features_dir: str = ".featureout",
    device: torch.device = torch.device("cpu"),
    min_match_count: int = 20,
    verbose: bool = True
) -> None:
    """
    使用LightGlue对特征进行匹配，并将匹配结果写入HDF5文件。

    Args:
        image_paths (List[str]): 图像路径列表。
        pair_indices (List[Tuple[int, int]]): 待匹配图像索引对。
        features_dir (str): 特征文件夹路径。
        device (torch.device): 设备。
        min_match_count (int): 最少匹配点数。
        verbose (bool): 是否打印匹配信息。
    """

    features_path = Path(features_dir)
    matcher = KF.LightGlueMatcher(
        "aliked",
        {
            "width_confidence": -1,
            "depth_confidence": -1,
            "mp": "cuda" in str(device)
        }
    ).eval().to(device)

    # 打开HDF5文件读取特征，写入匹配结果
    with h5py.File(str(features_path / "keypoints_rot.h5"), "r") as f_kp_rot, \
         h5py.File(str(features_path / "descriptors.h5"), "r") as f_desc, \
         h5py.File(str(features_path / "matches.h5"), "w") as f_match:

        for idx1, idx2 in tqdm(pair_indices, desc="Matching pairs"):
            fname1, fname2 = image_paths[idx1], image_paths[idx2]
            key1 = Path(fname1).name
            key2 = Path(fname2).name

            try:
                kp1_rot = torch.from_numpy(f_kp_rot[key1][...]).to(device)
                kp2_rot = torch.from_numpy(f_kp_rot[key2][...]).to(device)
                desc1 = torch.from_numpy(f_desc[key1][...]).to(device)
                desc2 = torch.from_numpy(f_desc[key2][...]).to(device)

                with torch.inference_mode():
                    distances, indices = matcher(
                        desc1, desc2,
                        KF.laf_from_center_scale_ori(kp1_rot[None]),
                        KF.laf_from_center_scale_ori(kp2_rot[None])
                    )

                match_count = len(indices)
                if match_count == 0:
                    continue

                if verbose:
                    print(f"{key1}-{key2}: {match_count} matches")

                group = f_match.require_group(key1)
                if match_count >= min_match_count:
                    group.create_dataset(
                        key2,
                        data=indices.detach().cpu().numpy().reshape(-1, 2)
                    )
            except Exception as e:
                print(f"Failed to match {key1} and {key2}: {e}")

def import_into_colmap(img_dir, feature_dir ='.featureout', database_path = 'colmap.db'):
    db = COLMAPDatabase.connect(database_path)
    db.create_tables()
    single_camera = False
    fname_to_id = add_keypoints(db, feature_dir, img_dir, '', 'simple-pinhole', single_camera)
    add_matches(
        db,
        feature_dir,
        fname_to_id,
    )
    db.commit()
    return

In [6]:
# Collect vital info from the dataset

@dataclasses.dataclass
class Prediction:
    image_id: str | None  # A unique identifier for the row -- unused otherwise. Used only on the hidden test set.
    dataset: str
    filename: str
    cluster_index: int | None = None
    rotation: np.ndarray | None = None
    translation: np.ndarray | None = None

# Set is_train=True to run the notebook on the training data.
# Set is_train=False if submitting an entry to the competition (test data is hidden, and different from what you see on the "test" folder).
is_train = False
data_dir = '/kaggle/input/image-matching-challenge-2025'
workdir = '/kaggle/working/result/'
os.makedirs(workdir, exist_ok=True)

if is_train:
    sample_submission_csv = os.path.join(data_dir, 'train_labels.csv')
else:
    sample_submission_csv = os.path.join(data_dir, 'sample_submission.csv')

samples = {}
competition_data = pd.read_csv(sample_submission_csv)
for _, row in competition_data.iterrows():
    # Note: For the test data, the "scene" column has no meaning, and the rotation_matrix and translation_vector columns are random.
    if row.dataset not in samples:
        samples[row.dataset] = []
    samples[row.dataset].append(
        Prediction(
            image_id=None if is_train else row.image_id,
            dataset=row.dataset,
            filename=row.image
        )
    )

for dataset in samples:
    print(f'Dataset "{dataset}" -> num_images={len(samples[dataset])}')

Dataset "ETs" -> num_images=22
Dataset "amy_gardens" -> num_images=200
Dataset "fbk_vineyard" -> num_images=163
Dataset "imc2023_haiper" -> num_images=54
Dataset "imc2023_heritage" -> num_images=209
Dataset "imc2023_theather_imc2024_church" -> num_images=76
Dataset "imc2024_dioscuri_baalshamin" -> num_images=138
Dataset "imc2024_lizard_pond" -> num_images=214
Dataset "pt_brandenburg_british_buckingham" -> num_images=225
Dataset "pt_piazzasanmarco_grandplace" -> num_images=168
Dataset "pt_sacrecoeur_trevi_tajmahal" -> num_images=225
Dataset "pt_stpeters_stpauls" -> num_images=200
Dataset "stairs" -> num_images=51


In [7]:
gc.collect()

max_images = None 
datasets_to_process = None 

if is_train:
        # max_images = 5
    
    	# Data from IMC 2023 and 2024.
    	# 'imc2024_dioscuri_baalshamin',
    	# 'imc2023_theather_imc2024_church',
    	# 'imc2023_heritage',
    	# 'imc2023_haiper',
    	# 'imc2024_lizard_pond',
    	# Crowdsourced PhotoTourism data.
    	# 'pt_stpeters_stpauls',
    	# 'pt_brandenburg_british_buckingham',
    	# 'pt_piazzasanmarco_grandplace',
    	# 'pt_sacrecoeur_trevi_tajmahal',
    
        datasets_to_process = [
        	# New data.
        	'amy_gardens',
        	'ETs',
        	'fbk_vineyard',
        	'stairs',
        ]

timings = {
    "shortlisting":[],
    "feature_detection": [],
    "feature_matching":[],
    "RANSAC": [],
    "Reconstruction": [],
}
mapping_result_strs = []


for dataset, predictions in samples.items():
    if datasets_to_process and dataset not in datasets_to_process:
        print(f'Skipping "{dataset}"')
        continue
    
    images_dir = os.path.join(data_dir, 'train' if is_train else 'test', dataset)
    images = [os.path.join(images_dir, p.filename) for p in predictions]
    if max_images is not None:
        images = images[:max_images]

    print(f'\nProcessing dataset "{dataset}": {len(images)} images')

    filename_to_index = {p.filename: idx for idx, p in enumerate(predictions)}

    feature_dir = os.path.join(workdir, 'featureout', dataset)
    os.makedirs(feature_dir, exist_ok=True)

    # Wrap algos in try-except blocks so we can populate a submission even if one scene crashes.
    try:
        t = time()
        index_pairs = get_image_pairs_shortlist(
            images,
            sim_th = 0.48, # should be strict
            min_pairs = 50, # we should select at least min_pairs PER IMAGE with biggest similarity
            exhaustive_if_less = 50,
            device=device
        )

        r1 = execute_rotation_detection(images, device)
        
        timings['shortlisting'].append(time() - t)
        print (f'Shortlisting. Number of pairs to match: {len(index_pairs)}. Done in {time() - t:.4f} sec')
        gc.collect()
    
        t = time()

        detect_aliked(images, r1, feature_dir, 6028, device=device)
        gc.collect()
        timings['feature_detection'].append(time() - t)
        print(f'Features detected in {time() - t:.4f} sec')
        
        t = time()
        match_with_lightglue(images, index_pairs, features_dir=feature_dir, device=device, verbose=False)
        timings['feature_matching'].append(time() - t)
        print(f'Features matched in {time() - t:.4f} sec')

        database_path = os.path.join(feature_dir, 'colmap.db')
        if os.path.isfile(database_path):
            os.remove(database_path)
        gc.collect()
        sleep(1)
        import_into_colmap(images_dir, feature_dir=feature_dir, database_path=database_path)
        output_path = f'{feature_dir}/colmap_rec_aliked'
        
        t = time()
        pycolmap.match_exhaustive(database_path)
        timings['RANSAC'].append(time() - t)
        print(f'Ran RANSAC in {time() - t:.4f} sec')
        
        # By default colmap does not generate a reconstruction if less than 10 images are registered.
        # Lower it to 3.
        mapper_options = pycolmap.IncrementalPipelineOptions()
        mapper_options.min_model_size = 8
        mapper_options.max_num_models = 35
        os.makedirs(output_path, exist_ok=True)
        t = time()
        maps = pycolmap.incremental_mapping(
            database_path=database_path, 
            image_path=images_dir,
            output_path=output_path,
            options=mapper_options)
        sleep(1)
        timings['Reconstruction'].append(time() - t)
        print(f'Reconstruction done in  {time() - t:.4f} sec')
        print(maps)

        clear_output(wait=False)
    
        registered = 0
        for map_index, cur_map in maps.items():
            for index, image in cur_map.images.items():
                prediction_index = filename_to_index[image.name]
                predictions[prediction_index].cluster_index = map_index
                predictions[prediction_index].rotation = deepcopy(image.cam_from_world.rotation.matrix())
                predictions[prediction_index].translation = deepcopy(image.cam_from_world.translation)
                registered += 1
        mapping_result_str = f'Dataset "{dataset}" -> Registered {registered} / {len(images)} images with {len(maps)} clusters'
        mapping_result_strs.append(mapping_result_str)
        print(mapping_result_str)
        gc.collect()
    except Exception as e:
        print(e)
        # raise e
        mapping_result_str = f'Dataset "{dataset}" -> Failed!'
        mapping_result_strs.append(mapping_result_str)
        print(mapping_result_str)

print('\nResults')
for s in mapping_result_strs:
    print(s)

print('\nTimings')
for k, v in timings.items():
    print(f'{k} -> total={sum(v):.02f} sec.')

Dataset "stairs" -> Registered 25 / 51 images with 3 clusters

Results
Dataset "ETs" -> Registered 41 / 22 images with 2 clusters
Dataset "amy_gardens" -> Failed!
Dataset "fbk_vineyard" -> Failed!
Dataset "imc2023_haiper" -> Failed!
Dataset "imc2023_heritage" -> Failed!
Dataset "imc2023_theather_imc2024_church" -> Failed!
Dataset "imc2024_dioscuri_baalshamin" -> Failed!
Dataset "imc2024_lizard_pond" -> Failed!
Dataset "pt_brandenburg_british_buckingham" -> Failed!
Dataset "pt_piazzasanmarco_grandplace" -> Failed!
Dataset "pt_sacrecoeur_trevi_tajmahal" -> Failed!
Dataset "pt_stpeters_stpauls" -> Failed!
Dataset "stairs" -> Registered 25 / 51 images with 3 clusters

Timings
shortlisting -> total=16.00 sec.
feature_detection -> total=5.99 sec.
feature_matching -> total=44.30 sec.
RANSAC -> total=16.37 sec.
Reconstruction -> total=91.60 sec.


In [8]:
# Must Create a submission file.
array_to_str = lambda array: ';'.join([f"{x:.09f}" for x in array])
none_to_str = lambda n: ';'.join(['nan'] * n)

submission_file = '/kaggle/working/submission.csv'
with open(submission_file, 'w') as f:
    if is_train:
        f.write('dataset,scene,image,rotation_matrix,translation_vector\n')
        for dataset in samples:
            for prediction in samples[dataset]:
                cluster_name = 'outliers' if prediction.cluster_index is None else f'cluster{prediction.cluster_index}'
                rotation = none_to_str(9) if prediction.rotation is None else array_to_str(prediction.rotation.flatten())
                translation = none_to_str(3) if prediction.translation is None else array_to_str(prediction.translation)
                f.write(f'{prediction.dataset},{cluster_name},{prediction.filename},{rotation},{translation}\n')
    else:
        f.write('image_id,dataset,scene,image,rotation_matrix,translation_vector\n')
        for dataset in samples:
            for prediction in samples[dataset]:
                cluster_name = 'outliers' if prediction.cluster_index is None else f'cluster{prediction.cluster_index}'
                rotation = none_to_str(9) if prediction.rotation is None else array_to_str(prediction.rotation.flatten())
                translation = none_to_str(3) if prediction.translation is None else array_to_str(prediction.translation)
                f.write(f'{prediction.image_id},{prediction.dataset},{cluster_name},{prediction.filename},{rotation},{translation}\n')

!head {submission_file}

image_id,dataset,scene,image,rotation_matrix,translation_vector
ETs_another_et_another_et001.png_public,ETs,cluster1,another_et_another_et001.png,0.555644962;-0.399619348;0.729083708;0.305174338;0.913734176;0.268250778;-0.773386902;0.073445444;0.629665361,-1.184348576;-1.218114300;1.053224445
ETs_another_et_another_et002.png_public,ETs,cluster1,another_et_another_et002.png,0.567277711;-0.398926305;0.720453886;0.319927362;0.912879888;0.253568122;-0.758842857;0.086649367;0.645483854,-1.036965855;-0.545002251;-0.167646122
ETs_another_et_another_et003.png_public,ETs,cluster1,another_et_another_et003.png,0.515627838;-0.430697023;0.740694274;0.382585232;0.889245019;0.250742571;-0.766652772;0.154088841;0.623297807,-1.280537155;0.814475971;-1.633580220
ETs_another_et_another_et004.png_public,ETs,cluster1,another_et_another_et004.png,0.536526589;-0.403890535;0.740953207;0.218858301;0.914582350;0.340059069;-0.815009364;-0.020286972;0.579092545,-1.175267336;-0.917724582;-1.329375497
ETs_another_e

In [9]:
print('    IMC 2025  Good! ')

    IMC 2025  Good! 
